# OMM 데이터 분석 — 2팀 (③ 사람 이름 · ④ 맛 표현)

**사용법**
1. 맨 위 `[0단계] 준비` 셀부터 차례로 실행 (Shift+Enter)
2. 분석 코드를 고치려면 `%%writefile` 셀 안에서 고친 뒤 그 셀을 실행 → `report/analyses/` 의 내 파일이 저장돼요
3. 바로 아래 `[실행]` 셀을 실행하면 고친 코드로 그림이 다시 그려지고, `[데이터 확인]` 셀에서 불러온 DataFrame 을 볼 수 있어요
4. 사이트 반영: 관리자 화면 › 보고서 차트 › **다시 그리기** (또는 매일 18:30 자동). 데이터 분석 탭의 코드 칸은 저장하는 즉시 바뀌어요

> 다른 팀 파일(`analyses/` 의 다른 번호)과 `common.py` 는 고치지 마세요. 공통 부분을 바꿔야 하면 팀에 먼저 알려 주세요.

In [ ]:
# [0단계] 준비 — 공통 코드(common.py) 불러오기 · DB 접속 정보
import os, sys, importlib
REPORT = os.getenv('OMM_REPORT', '/opt/pipeline/api/report')   # back1 기준 분석 코드 폴더
sys.path.insert(0, REPORT)

# DB 접속: API 와 같은 읽기 전용 계정. 비밀번호는 화면에 출력하지 않아요
if not os.getenv('PGPASSWORD'):
    try:   # 설치 때 만든 읽기 전용 DB 접속 파일 (팀원 계정만 읽을 수 있음)
        for line in open(os.path.join(REPORT, '.pgenv'), encoding='utf-8'):
            k, _, v = line.strip().partition('=')
            if k.startswith('PG') and v: os.environ.setdefault(k, v.strip('"\''))
    except (FileNotFoundError, PermissionError):
        from getpass import getpass
        os.environ['PGPASSWORD'] = getpass('DB 비밀번호 (omm_api): ')

# 노트북에서 그린 그림은 notebooks/_out 에 저장 (사이트 그림은 건드리지 않음)
os.environ['REPORT_DIR'] = os.path.join(REPORT, 'notebooks', '_out')
os.makedirs(os.environ['REPORT_DIR'], exist_ok=True)

import common
common.SHOW = True          # save() 할 때 그림을 셀 아래에 바로 보여 주기
from common import *
print('준비 완료')

## ③ 사람 이름
레시피 제목의 사람 이름 포함 여부와 조회수 · 별점의 관계

- 파일: `report/analyses/a3_person.py` · 함수: `chart3()`
- 그림 파일: `notebooks/_out/` 에 PNG · SVG 로 저장

In [ ]:
%%writefile ../analyses/a3_person.py
"""③ 사람 이름 — 제목의 사람 이름 포함 여부 × 조회수 · 별점
담당: 2팀 (③④) · 이 파일만 고치면 돼요.
  - 노트북 notebooks/2팀_③④.ipynb 에서 고치고 바로 실행해 볼 수 있어요.
  - 서버는 매일 18:30 charts_v4.py 가 이 파일의 chart3() 을 불러 그림을 다시 그려요.
  - 관리자 화면 › 데이터 분석 탭의 '차트 그리기' 코드 칸에 이 파일 내용이 그대로 보여요.
"""
from common import *

# [1단계] DB에서 데이터 불러오기 (q = SQL → pandas DataFrame)
# [2단계] pandas 로 가공
# [3단계] matplotlib 으로 그리고 save() 로 저장 (PNG + SVG)

def chart3():
    d = q("""SELECT (rm.person_name IS NOT NULL) AS p, rm.person_name AS nm, r.view_count AS v, r.avg_rating AS r
             FROM omm_recipes r JOIN omm_recipe_metrics rm USING (recipe_id) WHERE r.view_count > 0""")
    if d.empty or d.p.nunique() < 2: return empty('3_사람이름.png', 3, '사람 이름 vs 조회수 · 별점', '사람 이름이 든 레시피가 없음')
    fig, axs = plt.subplots(1, 3, figsize=(15, 5.5), gridspec_kw={'width_ratios': [1.3, 1, 1]})
    grp = [d[~d.p].v, d[d.p].v]; lab = [f'이름 없음\n(n={len(grp[0]):,})', f'이름 있음\n(n={len(grp[1]):,})']
    bp = axs[0].boxplot(grp, tick_labels=lab, patch_artist=True, widths=0.5, showfliers=False, medianprops={'color': INK, 'lw': 2})
    for p, c in zip(bp['boxes'], [S[0], S[1]]): p.set_facecolor(c); p.set_alpha(0.5); p.set_edgecolor(c)
    for i, g in enumerate(grp, 1): axs[0].text(i + 0.3, g.median(), f'중앙 {g.median():,.0f}', va='center', fontsize=11, color=INK2)
    axs[0].set_yscale('log'); logticks(axs[0].yaxis, [100, 1000, 10000, 100000, 1000000], lambda v, p: f'{v:,.0f}'); axs[0].set_title('③ 조회수 (로그)')
    m = [d[~d.p].r.mean(), d[d.p].r.mean()]
    axs[1].bar(['이름 없음', '이름 있음'], m, color=[S[0], S[1]], width=0.5)
    for i, v in enumerate(m): axs[1].text(i, v, f'★{v:.2f}', ha='center', va='bottom', color=INK)
    axs[1].set_ylim(max(0, min(m) - 0.8), 5.2); axs[1].set_title('평균 별점'); axs[1].grid(axis='x', visible=False)
    t = d[d.p].groupby('nm').agg(n=('v', 'size'), v=('v', 'median')).sort_values('n').tail(10)
    axs[2].barh(t.index, t.n, color=S[1], height=0.6)
    for y, (n, v) in enumerate(zip(t.n, t.v)): axs[2].text(n, y, f'  {n}개 · 중앙 {v:,.0f}회', va='center', fontsize=11, color=INK2)
    axs[2].set_title('많이 나온 이름'); axs[2].grid(axis='y', visible=False)
    note(axs[0], '사람 이름 = 제목에 백종원 · 류수영 등 사전에 있는 이름이 들어간 레시피')
    save(fig, '3_사람이름.png', 3, '사람 이름 vs 조회수 · 별점', '이름 있는 레시피와 없는 레시피 비교')

if __name__ == '__main__':
    chart3()

In [ ]:
# [실행] 저장한 파일을 다시 읽어서 그림 그리기
from analyses import a3_person
importlib.reload(a3_person)
common.queries()          # 이전 기록 비우기
a3_person.chart3()

In [ ]:
# [데이터 확인] ③ 사람 이름 이 DB 에서 불러온 DataFrame (SQL → pandas)
for i, (sql, df) in enumerate(common.queries(), 1):
    print(f'── 불러온 데이터 {i} · {len(df):,}행 × {df.shape[1]}열')
    display(df.head(10))

## ④ 맛 표현
맛 표현별 주요 재료 집계 → "매콤한 거 먹고 싶어" 같은 상황 문장 추천에 활용

- 파일: `report/analyses/a4_taste.py` · 함수: `chart4()`
- 그림 파일: `notebooks/_out/` 에 PNG · SVG 로 저장

In [ ]:
%%writefile ../analyses/a4_taste.py
"""④ 맛 표현 — 맛 표현별 주요 재료 → "매콤한 거 먹고 싶어" 같은 문장 추천
담당: 2팀 (③④) · 이 파일만 고치면 돼요.
  - 노트북 notebooks/2팀_③④.ipynb 에서 고치고 바로 실행해 볼 수 있어요.
  - 서버는 매일 18:30 charts_v4.py 가 이 파일의 chart4() 을 불러 그림을 다시 그려요.
  - 관리자 화면 › 데이터 분석 탭의 '차트 그리기' 코드 칸에 이 파일 내용이 그대로 보여요.
"""
from common import *

# [1단계] DB에서 데이터 불러오기 (q = SQL → pandas DataFrame)
# [2단계] pandas 로 가공
# [3단계] matplotlib 으로 그리고 save() 로 저장 (PNG + SVG)

def chart4():
    d = q("""SELECT t.taste_group AS g, i.name AS ing, count(DISTINCT rt.recipe_id) AS n,
                    (SELECT count(DISTINCT rt2.recipe_id) FROM omm_recipe_tastes rt2 JOIN omm_taste_words t2 USING (taste_id) WHERE t2.taste_group = t.taste_group) AS tot
             FROM omm_recipe_tastes rt JOIN omm_taste_words t USING (taste_id) JOIN omm_recipe_ingredients ri USING (recipe_id)
             JOIN omm_ingredients i ON i.ingredient_id = ri.ingredient_id AND NOT i.is_seasoning GROUP BY t.taste_group, i.name""")
    if d.empty: return empty('4_맛표현_재료.png', 4, '맛 표현별 주요 재료', '맛 표현이 든 레시피가 없음')
    d['pct'] = d.n / d.tot * 100
    cols = d.groupby('ing').n.sum().sort_values(ascending=False).head(14).index
    m = d.pivot_table(index='g', columns='ing', values='pct', fill_value=0).reindex(columns=cols, fill_value=0)
    m = m.loc[d.groupby('g').tot.first().sort_values(ascending=False).index]
    fig, ax = plt.subplots(figsize=(max(9, len(cols) * 0.85), 1.2 + len(m) * 0.7))
    im = ax.imshow(m.values, cmap='Blues', aspect='auto', vmin=0)
    ax.set_xticks(range(len(cols)), cols, rotation=40, ha='right'); ax.set_yticks(range(len(m)), [f'{g} ({int(d[d.g == g].tot.iloc[0]):,})' for g in m.index])
    ax.grid(False)
    for y in range(m.shape[0]):
        for x in range(m.shape[1]):
            v = m.values[y, x]
            if v >= 5: ax.text(x, y, f'{v:.0f}', ha='center', va='center', fontsize=11, color='white' if v > m.values.max() * 0.6 else INK)
    ax.set_title('④ 맛 표현별 주요 재료 — 그 맛 레시피 중 재료가 들어간 비율(%)')
    fig.colorbar(im, ax=ax, shrink=0.8, label='%')
    note(ax, '맛 표현 = 제목 · 소개 · 후기에서 찾은 단어 (매콤 · 얼큰 → 매운맛 …) · 괄호 = 레시피 수')
    save(fig, '4_맛표현_재료.png', 4, '맛 표현별 주요 재료', '"매콤한 거" → 어떤 재료가 들어간 메뉴인지')

if __name__ == '__main__':
    chart4()

In [ ]:
# [실행] 저장한 파일을 다시 읽어서 그림 그리기
from analyses import a4_taste
importlib.reload(a4_taste)
common.queries()          # 이전 기록 비우기
a4_taste.chart4()

In [ ]:
# [데이터 확인] ④ 맛 표현 이 DB 에서 불러온 DataFrame (SQL → pandas)
for i, (sql, df) in enumerate(common.queries(), 1):
    print(f'── 불러온 데이터 {i} · {len(df):,}행 × {df.shape[1]}열')
    display(df.head(10))

---
### 결과 정리 (직접 작성)
- 분석 결과:
- 서비스에 활용한 곳:
- 수익 모델과의 연결: